In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from google.colab import files
import os

# ==============================================================================
# 1. ЗАГРУЗКА ДАННЫХ
# ==============================================================================
# ПРИМЕЧАНИЕ: Если файлы уже лежат в Colab, скрипт не будет просить загружать их заново
if not os.path.exists('train.csv') or not os.path.exists('test.csv'):
    print("Загрузите файлы train.csv и test.csv для нового турнира:")
    uploaded = files.upload()

print("Читаем датасеты...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')




Читаем датасеты...


In [3]:
train

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699955,699955,715.39,4.10,3,3.24,45,31,1598.50,0,20.8,...,3D Secure,POS,Windows PC,False,False,False,False,False,False,0.0
699956,699956,32.28,0.04,6,8.56,44,81,2149.48,1,65.0,...,No Authentication,Online,iPhone,False,False,False,False,False,False,0.0
699957,699957,22.95,4.99,4,4.23,37,69,959.88,0,19.3,...,Biometric,Online,Android Phone,False,False,True,False,False,False,0.0
699958,699958,14.19,1.86,4,28.08,40,72,1362.85,0,31.0,...,PIN,ATM,Windows PC,False,False,True,False,False,False,0.0


In [4]:
import numpy as np
import pandas as pd
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler

def apply_dailycodepython_methodology(df):
    """
    Продвинутый математический пайплайн инженерии признаков,
    полностью адаптированный под структуру данных соревнования Kaggle S6E11.
    """
    print("🛸 Запуск продвинутого физико-математического пайплайна...")
    df_feat = df.copy()

    # =========================================================================
    # 1. Сжатие Лоренца для финансовых сумм (Lorentz Amount Damping)
    # =========================================================================
    if 'amount_usd' in df_feat.columns:
        print("-> Применяем релятивистское сжатие Лоренца к суммам транзакций...")
        df_feat['amount_usd'] = df_feat['amount_usd'].fillna(df_feat['amount_usd'].median())
        scale_factor = df_feat['amount_usd'].median() if df_feat['amount_usd'].median() > 0 else 1.0
        df_feat['lorentz_compressed_amount'] = df_feat['amount_usd'] / np.sqrt(1.0 + (df_feat['amount_usd'] / scale_factor) ** 2)

    # =========================================================================
    # 2. Релятивистский фактор плотности времени (Velocity Damping Factor)
    # =========================================================================
    # Используем готовую колонку hours_since_last_txn. Переводим часы в минуты для масштаба.
    if 'hours_since_last_txn' in df_feat.columns:
        print("-> Расчет релятивистского фактора временной плотности...")
        df_feat['hours_since_last_txn'] = df_feat['hours_since_last_txn'].fillna(df_feat['hours_since_last_txn'].median())
        minutes_since_last_txn = df_feat['hours_since_last_txn'] * 60.0
        df_feat['velocity_damping_factor'] = 1.0 / np.sqrt(1.0 + (minutes_since_last_txn / 60.0))

    # =========================================================================
    # 3. Геометрическая энтропия Шеннона (Оценка латентного хаоса профиля транзакции)
    # =========================================================================
    # Выбираем числовые поведенческие фичи для оценки системного сбоя в поведении
    behavioral_cols = [
        'txn_count_last_24h', 'distance_from_home_km', 'card_age_months',
        'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score'
    ]

    # Проверяем, что все колонки на месте
    behavioral_cols = [col for col in behavioral_cols if col in df_feat.columns]

    if len(behavioral_cols) > 0:
        print(f"-> Вычисление построчной энтропии Шеннона по поведенческим признакам ({len(behavioral_cols)} колонок)...")

        # Так как шкалы признаков разные (баланс в тысячах, а попытки CVV от 0 до 3),
        # временно нормализуем их перед расчетом энтропии
        scaler = StandardScaler()
        scaled_behavior = scaler.fit_transform(df_feat[behavioral_cols].fillna(0))

        def calculate_transaction_entropy(row):
            # Квантуем нормализованный вектор транзакции на 10 стабильных корзин
            counts, _ = np.histogram(row, bins=10, range=(-3, 3))
            total = counts.sum()
            if total == 0:
                return 0.0
            probs = counts / total
            return entropy(probs, base=2)

        # Считаем энтропию по строкам нормализованной матрицы
        df_feat['shannon_entropy_behavior'] = np.apply_along_axis(calculate_transaction_entropy, axis=1, arr=scaled_behavior)

    # =========================================================================
    # 4. Синергетические взаимодействия признаков (Fraud Feature Synergy)
    # =========================================================================
    if 'lorentz_compressed_amount' in df_feat.columns and 'velocity_damping_factor' in df_feat.columns:
        print("-> Синтез синергетических признаков финансовой угрозы...")
        df_feat['financial_velocity_synergy'] = df_feat['lorentz_compressed_amount'] * df_feat['velocity_damping_factor']

        if 'shannon_entropy_behavior' in df_feat.columns:
            df_feat['chaos_velocity_synergy'] = df_feat['shannon_entropy_behavior'] * df_feat['velocity_damping_factor']

    print(f"🎉 Пайплайн успешно выполнен! Добавлены новые робастные признаки. Новый размер: {df_feat.shape}\n")
    return df_feat


In [5]:
# НАСТРОЙКА ПОД ТУРНИР S6E11 (Credit Card Fraud)
# TODO: Обязательно проверь точное имя таргета в скачанном файле (обычно 'Class')
target_col = 'is_fraud'
id_col = 'id'

# Сохраняем таргет и id, чтобы они не участвовали в генерации признаков
y_train = train[target_col]
train_id = train[id_col]
test_id = test[id_col]

# Создаем чистые матрицы признаков для объединения
X_train = train.drop(columns=[id_col, target_col])
X_test = test.drop(columns=[id_col])

# Объединяем для сквозного расчета математических фич
df_all = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)

# ==============================================================================
# 2. ПРИМЕНЕНИЕ ТВОЕЙ МЕТОДОЛОГИИ
# ==============================================================================
# Вызываем ранее написанную функцию (она модифицирует df_all)
df_all_engineered = apply_dailycodepython_methodology(df_all)

# Разделяем датасет обратно на Train и Test после инженерии признаков
X_train_final = df_all_engineered.iloc[:len(train)].copy()
X_test_final = df_all_engineered.iloc[len(train):].copy()

print(f"Готово к обучению! Матрица Train: {X_train_final.shape}, Матрица Test: {X_test_final.shape}")

🛸 Запуск продвинутого физико-математического пайплайна...
-> Применяем релятивистское сжатие Лоренца к суммам транзакций...
-> Расчет релятивистского фактора временной плотности...
-> Вычисление построчной энтропии Шеннона по поведенческим признакам (7 колонок)...
-> Синтез синергетических признаков финансовой угрозы...
🎉 Пайплайн успешно выполнен! Добавлены новые робастные признаки. Новый размер: (999944, 29)

Готово к обучению! Матрица Train: (699960, 29), Матрица Test: (299984, 29)


In [6]:
# ==============================================================================
# 2. ПОДГОТОВКА КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ И МАТРИЦ
# ==============================================================================
# Автоматически находим все текстовые и логические колонки (кроме ID и таргета)
categorical_features = X_train_final.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()

# Переводим их в специальный тип 'category', который LightGBM понимает без OHE
for col in categorical_features:
    X_train_final[col] = X_train_final[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

print(f"Обнаружено категориальных признаков: {len(categorical_features)}")

# ==============================================================================
# 3. ОБУЧЕНИЕ LIGHTGBM НА 5 ФОЛДАХ
# ==============================================================================
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1
}

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_final))
test_preds = np.zeros(len(X_test_final))

print("\n🚀 Начало обучения моделей на 5 фолдах...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train_final, y_train)):
    # Вырезаем фолды из подготовленных финальных матриц
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[val_idx], y_train.iloc[val_idx]

    # Создаем датасеты LightGBM с указанием категорий
    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_features)
    val_data = lgb.Dataset(X_va, label=y_va, reference=train_data, categorical_feature=categorical_features)

    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=3000,
        valid_sets=[train_data, val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    # Предсказания
    oof_preds[val_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test_final, num_iteration=model.best_iteration) / folds.n_splits

    fold_auc = roc_auc_score(y_va, oof_preds[val_idx])
    print(f"--- Фолд {fold + 1} завершен. AUC: {fold_auc:.5f} ---")

cv_auc = roc_auc_score(y_train, oof_preds)
print(f"\n🏆 ИТОГОВЫЙ СРЕДНИЙ СКОР КРОСС-ВАЛИДАЦИИ (OOF AUC): {cv_auc:.5f}")

# ==============================================================================
# 4. СБОРКА ФИНАЛЬНОГО ФАЙЛА (С КОРРЕКТНЫМ ИМЕНЕМ ТАРГЕТА)
# ==============================================================================
submission = pd.DataFrame({
    'id': test_id.astype(int),
    'is_fraud': test_preds  # Изменено с 'satisfaction' на 'is_fraud'
})

submission.to_csv('submission.csv', index=False)
print("\n💾 Файл 'submission.csv' успешно создан и готов к отправке!")


Обнаружено категориальных признаков: 11

🚀 Начало обучения моделей на 5 фолдах...
[200]	training's auc: 0.962015	valid_1's auc: 0.944015
--- Фолд 1 завершен. AUC: 0.94418 ---
[200]	training's auc: 0.96152	valid_1's auc: 0.946923
[400]	training's auc: 0.975077	valid_1's auc: 0.947099
--- Фолд 2 завершен. AUC: 0.94736 ---
[200]	training's auc: 0.962083	valid_1's auc: 0.95033
--- Фолд 3 завершен. AUC: 0.95036 ---
[200]	training's auc: 0.962031	valid_1's auc: 0.946041
--- Фолд 4 завершен. AUC: 0.94614 ---
[200]	training's auc: 0.963991	valid_1's auc: 0.939825
[400]	training's auc: 0.976293	valid_1's auc: 0.940192
--- Фолд 5 завершен. AUC: 0.94037 ---

🏆 ИТОГОВЫЙ СРЕДНИЙ СКОР КРОСС-ВАЛИДАЦИИ (OOF AUC): 0.94556

💾 Файл 'submission.csv' успешно создан и готов к отправке!


In [7]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 73.6 MB/s  0:00:01


In [8]:
from catboost import CatBoostClassifier, Pool
# ==============================================================================
# 4. ОБУЧЕНИЕ CATBOOST НА 5 ФОЛДАХ
# ==============================================================================
# Автоматически находим ВСЕ категориальные признаки для CatBoost
# (Они уже имеют тип 'category' после шага с LightGBM)
categorical_features = X_train_final.select_dtypes(include=['category', 'object', 'bool']).columns.tolist()
print(f"Зафиксированы категориальные признаки для CatBoost: {categorical_features}")

cat_params = {
    'iterations': 3000,
    'learning_rate': 0.05,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'task_type': 'GPU',            # Переключили на видеокарту
    'cat_features': categorical_features, # Явно дублируем категории для GPU-движка
    'early_stopping_rounds': 100,
    'verbose': 200
}

oof_preds_cat = np.zeros(len(X_train_final))
test_preds_cat = np.zeros(len(X_test_final))

print("\n🚀 Начало обучения CatBoost на 5 фолдах...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train_final, y_train)):
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[val_idx], y_train.iloc[val_idx]

    # Для CatBoost упаковываем данные в Pool с явным указанием категорий
    train_pool = Pool(X_tr, y_tr, cat_features=categorical_features)
    val_pool = Pool(X_va, y_va, cat_features=categorical_features)

    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(train_pool, eval_set=val_pool, use_best_model=True)

    # Предсказания вероятностей класса 1
    oof_preds_cat[val_idx] = model_cat.predict_proba(val_pool)[:, 1]

    test_pool = Pool(X_test_final, cat_features=categorical_features)
    test_preds_cat += model_cat.predict_proba(test_pool)[:, 1] / folds.n_splits

    fold_auc = roc_auc_score(y_va, oof_preds_cat[val_idx])
    print(f"-> CatBoost Фолд {fold + 1} завершен. AUC: {fold_auc:.5f}")

cv_auc_cat = roc_auc_score(y_train, oof_preds_cat)
print(f"\n🏆 ИТОГОВЫЙ OOF AUC для CatBoost: {cv_auc_cat:.5f}")


Зафиксированы категориальные признаки для CatBoost: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt']

🚀 Начало обучения CatBoost на 5 фолдах...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8159105	best: 0.8159105 (0)	total: 81.3ms	remaining: 4m 3s
200:	test: 0.9401066	best: 0.9401112 (198)	total: 10.4s	remaining: 2m 25s
400:	test: 0.9417885	best: 0.9417885 (400)	total: 21s	remaining: 2m 16s
600:	test: 0.9423396	best: 0.9423396 (600)	total: 30.5s	remaining: 2m 1s
800:	test: 0.9424533	best: 0.9424745 (711)	total: 41.1s	remaining: 1m 52s
1000:	test: 0.9427253	best: 0.9427364 (984)	total: 51.5s	remaining: 1m 42s
1200:	test: 0.9428007	best: 0.9428212 (1149)	total: 1m 2s	remaining: 1m 33s
bestTest = 0.9428212047
bestIteration = 1149
Shrink model to first 1150 iterations.
-> CatBoost Фолд 1 завершен. AUC: 0.94282


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7840419	best: 0.7840419 (0)	total: 103ms	remaining: 5m 9s
200:	test: 0.9421686	best: 0.9421686 (200)	total: 10.6s	remaining: 2m 28s
400:	test: 0.9439032	best: 0.9439032 (400)	total: 21.1s	remaining: 2m 16s
600:	test: 0.9449655	best: 0.9449717 (596)	total: 31.9s	remaining: 2m 7s
800:	test: 0.9453274	best: 0.9453274 (800)	total: 42.8s	remaining: 1m 57s
1000:	test: 0.9457244	best: 0.9457244 (1000)	total: 53.6s	remaining: 1m 46s
1200:	test: 0.9459347	best: 0.9459347 (1200)	total: 1m 3s	remaining: 1m 35s
1400:	test: 0.9460836	best: 0.9460920 (1391)	total: 1m 14s	remaining: 1m 24s
1600:	test: 0.9462062	best: 0.9462062 (1600)	total: 1m 25s	remaining: 1m 14s
1800:	test: 0.9463376	best: 0.9463415 (1794)	total: 1m 36s	remaining: 1m 4s
2000:	test: 0.9463975	best: 0.9464234 (1945)	total: 1m 47s	remaining: 53.5s
bestTest = 0.9464234114
bestIteration = 1945
Shrink model to first 1946 iterations.
-> CatBoost Фолд 2 завершен. AUC: 0.94642


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7833830	best: 0.7833830 (0)	total: 65.2ms	remaining: 3m 15s
200:	test: 0.9473023	best: 0.9473023 (200)	total: 10.8s	remaining: 2m 30s
400:	test: 0.9485340	best: 0.9485393 (399)	total: 21.7s	remaining: 2m 20s
600:	test: 0.9489807	best: 0.9489847 (597)	total: 32.7s	remaining: 2m 10s
800:	test: 0.9493597	best: 0.9493608 (799)	total: 43.6s	remaining: 1m 59s
bestTest = 0.949521184
bestIteration = 894
Shrink model to first 895 iterations.
-> CatBoost Фолд 3 завершен. AUC: 0.94952


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7700566	best: 0.7700566 (0)	total: 65.7ms	remaining: 3m 16s
200:	test: 0.9417431	best: 0.9417431 (200)	total: 10.7s	remaining: 2m 29s
400:	test: 0.9433206	best: 0.9433250 (399)	total: 21.7s	remaining: 2m 20s
600:	test: 0.9439171	best: 0.9439171 (600)	total: 31.7s	remaining: 2m 6s
800:	test: 0.9441788	best: 0.9441841 (799)	total: 42.4s	remaining: 1m 56s
1000:	test: 0.9442662	best: 0.9442840 (989)	total: 53.4s	remaining: 1m 46s
bestTest = 0.9442840219
bestIteration = 989
Shrink model to first 990 iterations.
-> CatBoost Фолд 4 завершен. AUC: 0.94428


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7840094	best: 0.7840094 (0)	total: 65.1ms	remaining: 3m 15s
200:	test: 0.9368550	best: 0.9368550 (200)	total: 10.9s	remaining: 2m 31s
400:	test: 0.9386972	best: 0.9386991 (394)	total: 20.9s	remaining: 2m 15s
600:	test: 0.9394960	best: 0.9394960 (600)	total: 31.2s	remaining: 2m 4s
800:	test: 0.9397969	best: 0.9398218 (792)	total: 41.9s	remaining: 1m 54s
1000:	test: 0.9399318	best: 0.9399372 (987)	total: 52.5s	remaining: 1m 44s
1200:	test: 0.9399502	best: 0.9400412 (1121)	total: 1m 3s	remaining: 1m 34s
bestTest = 0.940041244
bestIteration = 1121
Shrink model to first 1122 iterations.
-> CatBoost Фолд 5 завершен. AUC: 0.94004

🏆 ИТОГОВЫЙ OOF AUC для CatBoost: 0.94456


In [9]:
from scipy.stats import rankdata

# Переводим предсказания теста в ранги и усредняем их
rank_lgb = rankdata(test_preds) / len(test_preds)
rank_cat = rankdata(test_preds_cat) / len(test_preds_cat)

# Финальный бленд (например, 50/50 или оптимизируй веса через Nelder-Mead)
final_sub_preds = 0.5 * rank_lgb + 0.5 * rank_cat

# Запись полностью защищенного от тряски сабмишна
submission = pd.DataFrame({
    'id': test_id.astype(int),
    'is_fraud': final_sub_preds
})
submission.to_csv('submission_blend.csv', index=False)
print("Финишировали! Робастный ранговый бленд готов.")


Финишировали! Робастный ранговый бленд готов.


In [10]:
# ==============================================================================
# 5. СОЗДАНИЕ ИТОГОВОГО ТРОЙНОГО АНСАМБЛЯ (TRIPLE ENSEMBLE)
# ==============================================================================
# Усредняем предсказания моделей с твоими весами (0.4 / 0.3 / 0.3)
# Примечание: Убедись, что переменная test_preds_xgb посчитана в цикле для XGBoost
final_test_preds = (0.4 * test_preds) + (0.3 * test_preds_cat) + (0.3 * test_preds_xgb)

# Сохраняем финальный победный сабмит под формат текущего турнира S6E11
submission_ensemble = pd.DataFrame({
    'id': test_id.astype(int),    # Используем сохраненный ранее test_id
    'is_fraud': final_test_preds  # Изменено с 'satisfaction' на 'is_fraud'
})

# Запись файла на диск
submission_ensemble.to_csv('submission_triple_ensemble.csv', index=False)
print("Финальный ансамблевый сабмит 'submission_triple_ensemble.csv' готов к отправке!")

# Автоматическое скачивание файла из Colab на компьютер
files.download('submission_triple_ensemble.csv')


NameError: name 'test_preds_xgb' is not defined

In [12]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# ==============================================================================
# 5. ОБУЧЕНИЕ XGBOOST НА 5 ФОЛДАХ (ИСПРАВЛЕННЫЙ)
# ==============================================================================

# Создаем копии матриц специально для XGBoost, чтобы не испортить исходные данные
X_train_xgb = X_train_final.copy()
X_test_xgb = X_test_final.copy()

# Исправление булевых категорий: переводим их в int (0 и 1)
for col in X_train_xgb.columns:
    # Если это булевая колонка или категория, содержащая True/False
    if X_train_xgb[col].dtype == 'bool' or (X_train_xgb[col].dtype.name == 'category' and X_train_xgb[col].cat.categories.dtype == 'bool'):
        print(f"-> Конвертируем булеву колонку {col} в целые числа (0/1) для XGBoost...")
        X_train_xgb[col] = X_train_xgb[col].astype(int)
        X_test_xgb[col] = X_test_xgb[col].astype(int)
    # Если это обычная категория (строковая), оставляем её как category, XGBoost её поймет
    elif X_train_xgb[col].dtype.name == 'category':
        X_train_xgb[col] = X_train_xgb[col].astype('category')
        X_test_xgb[col] = X_test_xgb[col].astype('category')

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'device': 'cuda',               # Работа на GPU T4
    'enable_categorical': True,     # Поддержка оставшихся строковых категорий
    'reg_alpha': 1.5,
    'reg_lambda': 5.0,
    'random_state': 42
}

oof_preds_xgb = np.zeros(len(X_train_xgb))
test_preds_xgb = np.zeros(len(X_test_xgb))

print("\n🚀 Запуск XGBoost на GPU...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train_xgb, y_train)):
    X_tr, y_tr = X_train_xgb.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train_xgb.iloc[val_idx], y_train.iloc[val_idx]

    model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=3000, early_stopping_rounds=100)

    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=200
    )

    oof_preds_xgb[val_idx] = model_xgb.predict_proba(X_va)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test_xgb)[:, 1] / folds.n_splits

    fold_auc = roc_auc_score(y_va, oof_preds_xgb[val_idx])
    print(f"-> XGBoost Фолд {fold + 1} завершен. AUC: {fold_auc:.5f}")

cv_auc_xgb = roc_auc_score(y_train, oof_preds_xgb)
print(f"\n🏆 ИТОГОВЫЙ OOF AUC для XGBoost: {cv_auc_xgb:.5f}")


-> Конвертируем булеву колонку is_foreign_transaction в целые числа (0/1) для XGBoost...
-> Конвертируем булеву колонку is_new_merchant в целые числа (0/1) для XGBoost...
-> Конвертируем булеву колонку used_vpn в целые числа (0/1) для XGBoost...
-> Конвертируем булеву колонку ip_country_mismatch в целые числа (0/1) для XGBoost...
-> Конвертируем булеву колонку billing_shipping_mismatch в целые числа (0/1) для XGBoost...
-> Конвертируем булеву колонку is_ai_generated_scam_attempt в целые числа (0/1) для XGBoost...

🚀 Запуск XGBoost на GPU...
[0]	validation_0-auc:0.87507
[200]	validation_0-auc:0.94330
[400]	validation_0-auc:0.94390
[459]	validation_0-auc:0.94386


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [09:04:22] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


-> XGBoost Фолд 1 завершен. AUC: 0.94403
[0]	validation_0-auc:0.88037
[200]	validation_0-auc:0.94702
[400]	validation_0-auc:0.94789
[522]	validation_0-auc:0.94778
-> XGBoost Фолд 2 завершен. AUC: 0.94800
[0]	validation_0-auc:0.88267
[200]	validation_0-auc:0.95030
[400]	validation_0-auc:0.95065
[435]	validation_0-auc:0.95056
-> XGBoost Фолд 3 завершен. AUC: 0.95077
[0]	validation_0-auc:0.87920
[200]	validation_0-auc:0.94522
[369]	validation_0-auc:0.94551
-> XGBoost Фолд 4 завершен. AUC: 0.94560
[0]	validation_0-auc:0.86841
[200]	validation_0-auc:0.93976
[400]	validation_0-auc:0.94081
[589]	validation_0-auc:0.94057
-> XGBoost Фолд 5 завершен. AUC: 0.94087

🏆 ИТОГОВЫЙ OOF AUC для XGBoost: 0.94575


In [13]:
import numpy as np
from sklearn.metrics import roc_auc_score
import pandas as pd
from google.colab import files

# ==============================================================================
# MASTER BLENDING ENGINE (МЕТОД МАСАИ КАВАМАТЫ)
# ==============================================================================

# # 1. Собираем все три OOF-предсказания в единую матрицу
# Убедись, что все три массива (oof_preds, oof_preds_xgb, oof_preds_cat) уже посчитаны!
oof_matrix = np.column_stack([oof_preds, oof_preds_xgb, oof_preds_cat])

# Запускаем оптимизатор HILL CLIMB (шаг 0.005 для максимальной точности)
def optimize_weights_hill_climb(oof_preds_matrix, y_true, iterations=1000, step=0.005):
    n_models = oof_preds_matrix.shape[1]
    best_weights = np.ones(n_models) / n_models

    initial_blend = np.dot(oof_preds_matrix, best_weights)
    best_score = roc_auc_score(y_true, initial_blend)

    print(f"[СТАРТ] AUC при равных долях (по 33%): {best_score:.6f}\n")

    for i in range(iterations):
        improved = False
        for model_idx in range(n_models):
            for change in [-step, step]:
                test_weights = best_weights.copy()
                test_weights[model_idx] += change

                if test_weights[model_idx] < 0:
                    continue

                test_weights /= np.sum(test_weights)

                current_blend = np.dot(oof_preds_matrix, test_weights)
                current_score = roc_auc_score(y_true, current_blend)

                if current_score > best_score:
                    best_score = current_score
                    best_weights = test_weights
                    improved = True
        if not improved:
            break

    print(f"[УСПЕХ] Оптимизация завершена на итерации {i}!")
    print(f"[МАКСИМАЛЬНЫЙ AUC] На кросс-валидации: {best_score:.6f}")
    return best_weights

# Находим идеальные пропорции
# ВАЖНО: Используем y_train, который мы выделили из оригинального train
best_blend_weights = optimize_weights_hill_climb(oof_matrix, y_train)

# Выводим веса
print("\n" + "="*30 + "\n ИДЕАЛЬНЫЙ РЕЦЕПТ АНСАМБЛЯ \n" + "="*30)
print(f" Вес LightGBM: {best_blend_weights[0]:.4f}")
print(f" Вес XGBoost:  {best_blend_weights[1]:.4f}")
print(f" Вес CatBoost: {best_blend_weights[2]:.4f}")
print("="*30 + "\n")

# # 2. Применяем веса на тестовой выборке
final_ensemble_test_preds = (
    best_blend_weights[0] * test_preds +
    best_blend_weights[1] * test_preds_xgb +
    best_blend_weights[2] * test_preds_cat
)

# # 3. Сохраняем и скачиваем честный сабмит под турнир S6E11
# Изменено имя таргета на 'is_fraud' и источника id на 'test_id'
submission_master = pd.DataFrame({
    'id': test_id.astype(int),
    'is_fraud': final_ensemble_test_preds
})

output_filename = 'submission_shannon_entropy_non_linear_scaled_triple_stack.csv'
submission_master.to_csv(output_filename, index=False)
print(f" Победный файл '{output_filename}' сгенерирован и готов к отправке на Kaggle!")

# Скачиваем сгенерированный файл
files.download(output_filename)


[СТАРТ] AUC при равных долях (по 33%): 0.946233

[УСПЕХ] Оптимизация завершена на итерации 30!
[МАКСИМАЛЬНЫЙ AUC] На кросс-валидации: 0.946310

 ИДЕАЛЬНЫЙ РЕЦЕПТ АНСАМБЛЯ 
 Вес LightGBM: 0.3400
 Вес XGBoost:  0.4837
 Вес CatBoost: 0.1763

 Победный файл 'submission_shannon_entropy_non_linear_scaled_triple_stack.csv' сгенерирован и готов к отправке на Kaggle!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>